# Model Monitoring & Observability

## Learning Objectives
- Understand production model monitoring
- Detect data drift and model degradation
- Implement alerting systems
- Log predictions for analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, List, Optional
from dataclasses import dataclass
import json
from pathlib import Path
from scipy import stats

print("Model monitoring module loaded!")

## 1. Why Monitor ML Models?

Models degrade in production due to:

- **Data Drift**: Input distribution changes
- **Concept Drift**: Relationship between features and target changes
- **Feature Issues**: Missing values, schema changes
- **Infrastructure Issues**: Latency, errors, failures

In [ ]:
@dataclass
class PredictionLog:
    """Log structure for model predictions."""
    timestamp: str
    request_id: str
    features: Dict
    prediction: float
    confidence: float
    latency_ms: float
    model_version: str
    
    def to_dict(self) -> Dict:
        return {
            "timestamp": self.timestamp,
            "request_id": self.request_id,
            "features": self.features,
            "prediction": self.prediction,
            "confidence": self.confidence,
            "latency_ms": self.latency_ms,
            "model_version": self.model_version
        }


class PredictionLogger:
    """Log predictions for monitoring."""
    
    def __init__(self, log_dir: str = "prediction_logs"):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)
        self.logs: List[PredictionLog] = []
    
    def log(self, log_entry: PredictionLog):
        """Add a prediction log entry."""
        self.logs.append(log_entry)
        
        # Write to file periodically
        if len(self.logs) >= 100:
            self.flush()
    
    def flush(self):
        """Write logs to disk."""
        if not self.logs:
            return
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_file = self.log_dir / f"predictions_{timestamp}.jsonl"
        
        with open(log_file, "a") as f:
            for log in self.logs:
                f.write(json.dumps(log.to_dict()) + "\n")
        
        self.logs = []
        print(f"Flushed logs to {log_file}")


# Demo
logger = PredictionLogger("../monitoring/logs")
print("PredictionLogger initialized!")

## 2. Data Drift Detection

In [ ]:
class DriftDetector:
    """Detect data drift using statistical tests."""
    
    def __init__(self, reference_data: np.ndarray, threshold: float = 0.05):
        """Initialize with reference data (training distribution)."""
        self.reference_data = reference_data
        self.threshold = threshold
        self.reference_stats = self._compute_stats(reference_data)
    
    def _compute_stats(self, data: np.ndarray) -> Dict:
        """Compute distribution statistics."""
        return {
            "mean": np.mean(data, axis=0),
            "std": np.std(data, axis=0),
            "min": np.min(data, axis=0),
            "max": np.max(data, axis=0)
        }
    
    def detect_drift(self, current_data: np.ndarray) -> Dict:
        """Detect drift using Kolmogorov-Smirnov test."""
        results = {
            "drift_detected": False,
            "features_with_drift": [],
            "p_values": []
        }
        
        n_features = self.reference_data.shape[1] if len(self.reference_data.shape) > 1 else 1
        
        for i in range(n_features):
            ref = self.reference_data[:, i] if n_features > 1 else self.reference_data
            cur = current_data[:, i] if n_features > 1 else current_data
            
            # KS test
            statistic, p_value = stats.ks_2samp(ref, cur)
            results["p_values"].append(p_value)
            
            if p_value < self.threshold:
                results["drift_detected"] = True
                results["features_with_drift"].append(i)
        
        return results


# Demo: Create reference and drifted data
np.random.seed(42)
reference_data = np.random.normal(0, 1, (1000, 5))
current_data_no_drift = np.random.normal(0, 1, (500, 5))
current_data_with_drift = np.random.normal(0.5, 1.5, (500, 5))  # Shifted distribution

detector = DriftDetector(reference_data)

# Test without drift
result_no_drift = detector.detect_drift(current_data_no_drift)
print("Without drift:", result_no_drift["drift_detected"])

# Test with drift
result_with_drift = detector.detect_drift(current_data_with_drift)
print("With drift:", result_with_drift["drift_detected"])
print("Features with drift:", result_with_drift["features_with_drift"])

## 3. Performance Monitoring

In [ ]:
class PerformanceMonitor:
    """Monitor model performance metrics over time."""
    
    def __init__(self, baseline_metrics: Dict[str, float]):
        self.baseline = baseline_metrics
        self.history: List[Dict] = []
        self.alert_thresholds = {
            "accuracy": 0.05,   # Alert if drops by 5%
            "latency_p99": 100  # Alert if > 100ms
        }
    
    def record(self, metrics: Dict[str, float], timestamp: str = None):
        """Record metrics snapshot."""
        entry = {
            "timestamp": timestamp or datetime.now().isoformat(),
            "metrics": metrics
        }
        self.history.append(entry)
        return self._check_alerts(metrics)
    
    def _check_alerts(self, metrics: Dict[str, float]) -> List[str]:
        """Check if any metrics require alerts."""
        alerts = []
        
        # Check accuracy degradation
        if "accuracy" in metrics and "accuracy" in self.baseline:
            degradation = self.baseline["accuracy"] - metrics["accuracy"]
            if degradation > self.alert_thresholds["accuracy"]:
                alerts.append(
                    f"ALERT: Accuracy dropped by {degradation:.2%} "
                    f"(baseline: {self.baseline['accuracy']:.2%}, "
                    f"current: {metrics['accuracy']:.2%})"
                )
        
        # Check latency
        if "latency_p99" in metrics:
            if metrics["latency_p99"] > self.alert_thresholds["latency_p99"]:
                alerts.append(
                    f"ALERT: High latency - P99: {metrics['latency_p99']:.0f}ms"
                )
        
        return alerts
    
    def get_summary(self) -> pd.DataFrame:
        """Get metrics summary over time."""
        if not self.history:
            return pd.DataFrame()
        
        rows = []
        for entry in self.history:
            row = {"timestamp": entry["timestamp"]}
            row.update(entry["metrics"])
            rows.append(row)
        
        return pd.DataFrame(rows)


# Demo
monitor = PerformanceMonitor(baseline_metrics={"accuracy": 0.95, "latency_p99": 50})

# Simulate metrics over time
for day in range(5):
    # Simulating gradual degradation
    accuracy = 0.95 - (day * 0.02)
    latency = 50 + (day * 20)
    
    alerts = monitor.record(
        {"accuracy": accuracy, "latency_p99": latency},
        timestamp=f"2024-01-{day+1:02d}"
    )
    
    if alerts:
        for alert in alerts:
            print(f"Day {day+1}: {alert}")

print("\nMetrics Summary:")
print(monitor.get_summary())

## 4. Alerting System

In [ ]:
from enum import Enum

class AlertSeverity(str, Enum):
    INFO = "info"
    WARNING = "warning"
    CRITICAL = "critical"


@dataclass
class Alert:
    """Alert structure."""
    severity: AlertSeverity
    message: str
    metric_name: str
    current_value: float
    threshold: float
    timestamp: str = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().isoformat()


class AlertManager:
    """Manage alerts and notifications."""
    
    def __init__(self):
        self.alerts: List[Alert] = []
        self.handlers = []
    
    def add_handler(self, handler):
        """Add notification handler."""
        self.handlers.append(handler)
    
    def fire_alert(self, alert: Alert):
        """Fire an alert and notify handlers."""
        self.alerts.append(alert)
        
        for handler in self.handlers:
            handler(alert)
    
    def get_recent_alerts(self, hours: int = 24) -> List[Alert]:
        """Get alerts from last N hours."""
        cutoff = datetime.now() - timedelta(hours=hours)
        return [
            a for a in self.alerts
            if datetime.fromisoformat(a.timestamp) > cutoff
        ]


# Demo: Console handler
def console_handler(alert: Alert):
    icon = {"info": "ℹ️", "warning": "⚠️", "critical": "🚨"}[alert.severity.value]
    print(f"{icon} [{alert.severity.value.upper()}] {alert.message}")


alert_manager = AlertManager()
alert_manager.add_handler(console_handler)

# Fire some alerts
alert_manager.fire_alert(Alert(
    severity=AlertSeverity.WARNING,
    message="Accuracy degradation detected",
    metric_name="accuracy",
    current_value=0.88,
    threshold=0.90
))

alert_manager.fire_alert(Alert(
    severity=AlertSeverity.CRITICAL,
    message="Model latency exceeds SLA",
    metric_name="latency_p99",
    current_value=250,
    threshold=100
))

## 5. Comprehensive Monitoring Dashboard

In [ ]:
class ModelMonitor:
    """Comprehensive model monitoring system."""
    
    def __init__(self, model_name: str, reference_data: np.ndarray):
        self.model_name = model_name
        self.drift_detector = DriftDetector(reference_data)
        self.prediction_logger = PredictionLogger(f"../monitoring/{model_name}")
        self.alert_manager = AlertManager()
        self.alert_manager.add_handler(console_handler)
        
        # Metrics storage
        self.request_count = 0
        self.error_count = 0
        self.latencies: List[float] = []
    
    def record_prediction(
        self,
        features: Dict,
        prediction: float,
        confidence: float,
        latency_ms: float,
        request_id: str = None
    ):
        """Record a prediction for monitoring."""
        self.request_count += 1
        self.latencies.append(latency_ms)
        
        log = PredictionLog(
            timestamp=datetime.now().isoformat(),
            request_id=request_id or f"req_{self.request_count}",
            features=features,
            prediction=prediction,
            confidence=confidence,
            latency_ms=latency_ms,
            model_version="1.0.0"
        )
        self.prediction_logger.log(log)
    
    def check_health(self) -> Dict:
        """Get current health status."""
        if not self.latencies:
            return {"status": "no_data"}
        
        return {
            "status": "healthy" if self.error_count / max(1, self.request_count) < 0.01 else "degraded",
            "request_count": self.request_count,
            "error_rate": self.error_count / max(1, self.request_count),
            "latency_p50": np.percentile(self.latencies, 50),
            "latency_p95": np.percentile(self.latencies, 95),
            "latency_p99": np.percentile(self.latencies, 99)
        }


# Demo
reference_data = np.random.normal(0, 1, (1000, 5))
monitor = ModelMonitor("classifier", reference_data)

# Simulate predictions
for i in range(50):
    monitor.record_prediction(
        features={"f1": np.random.randn(), "f2": np.random.randn()},
        prediction=np.random.randint(0, 2),
        confidence=np.random.uniform(0.7, 1.0),
        latency_ms=np.random.uniform(10, 80)
    )

print("\nHealth Check:")
health = monitor.check_health()
for k, v in health.items():
    print(f"  {k}: {v}")

## Summary

### Key Monitoring Components

1. **Prediction Logging**: Track all predictions for analysis
2. **Drift Detection**: Statistical tests to detect distribution changes
3. **Performance Monitoring**: Track accuracy, latency, errors
4. **Alerting**: Automated notifications when thresholds exceeded
5. **Health Checks**: Overall system status

In [ ]:
print("=" * 50)
print("MODEL MONITORING COMPLETE")
print("=" * 50)
print("\nKey Components:")
print("  📊 PredictionLogger - Log all predictions")
print("  📈 DriftDetector - Detect data drift")
print("  ⚡ PerformanceMonitor - Track metrics")
print("  🚨 AlertManager - Handle alerts")
print("  🔍 ModelMonitor - Comprehensive monitoring")